In [1]:
import tensorflow as tf
tf.__version__


'2.20.0'

In [2]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Embedding , LSTM
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
vocab_size = 10000
max_len = 200


In [4]:
(x_train , y_train) , (x_test , y_test) = imdb.load_data(num_words=vocab_size)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
x_train = pad_sequences(x_train , maxlen=max_len)
x_test = pad_sequences(x_test , maxlen=max_len)

In [6]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size , output_dim=128 , input_length=max_len))
model.add(LSTM(128))
model.add(Dense(units=1 , activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [7]:
model.compile(loss="binary_crossentropy" , optimizer="adam" , metrics = ["accuracy"])

In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(x_train , y_train , epochs = 2 , batch_size=64 ,  validation_data =(x_test , y_test ))

Epoch 1/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 227s 576ms/step - accuracy: 0.7972 - loss: 0.4375 - val_accuracy: 0.8538 - val_loss: 0.3467
Epoch 2/2
391/391 ━━━━━━━━━━━━━━━━━━━━ 224s 572ms/step - accuracy: 0.8957 - loss: 0.2631 - val_accuracy: 0.8628 - val_loss: 0.3657


In [10]:
loss , accuracy = model.evaluate(x_test , y_test)
print("Accuracy : " , accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 73s 93ms/step - accuracy: 0.8628 - loss: 0.3657
Accuracy :  0.8628000020980835


In [11]:
model.save("lstm_model.h5")

In [12]:
from tensorflow.keras.models import load_model
model = load_model("lstm_model.h5")

In [13]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.datasets import imdb

# Get the original word index from the IMDb dataset
imdb_word_index = imdb.get_word_index()

# Adjust the word index to match how imdb.load_data preprocesses:
# 0 is padding, 1 is start of sequence, 2 is unknown, 3 is unused.
# Actual words start from index 4.
word_to_id = {k: (v + 3) for k, v in imdb_word_index.items()}
word_to_id["<PAD>"] = 0
word_to_id["<START>"] = 1
word_to_id["<UNK>"] = 2
word_to_id["<UNUSED>"] = 3

# Create a Tokenizer instance
# `num_words` ensures that only the top `vocab_size-1` words are considered, plus one for OOV.
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")

# Assign the adjusted word_index to the tokenizer
# Filter word_to_id to only include indices within vocab_size, consistent with model training
filtered_word_index = {word: idx for word, idx in word_to_id.items() if idx < vocab_size}
tokenizer.word_index = filtered_word_index

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [14]:
new_text = ["This movie is amazing", "Worst experience ever"]

from tensorflow.keras.preprocessing.sequence import pad_sequences

sequences = tokenizer.texts_to_sequences(new_text)
padded = pad_sequences(sequences, maxlen=100)  # same max_len as training

In [15]:
predictions = model.predict(padded)

for text, pred in zip(new_text, predictions):
    print(text, "->", "Positive" if pred > 0.5 else "Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
This movie is amazing -> Positive
Worst experience ever -> Negative
